In [3]:
from __future__ import annotations

from pathlib import Path
from typing import List, Tuple, Dict, Optional
from collections import defaultdict
import json
import numpy as np
from scipy.signal import welch

# ---------------- CONFIG ----------------
ROOT_DIR    = Path(r"/home/tsultan1/IROS/Data")
DATASET_DIR = ROOT_DIR / "_dataset_icml_v1"

# ---- Choose which Phase-5 export to read ----
QC_POLICY = "core_ok"   # must match Phase-5 run ("core_ok"/"tri_good"/...)
MODE      = "balanced"  # "balanced" or "ssl"

# Phase-5 version
P5_VER = "v6_3c_commitmeta"   # ✅ IMPORTANT
EXPORT_PREFIX  = f"exports_{P5_VER}_{MODE}_{QC_POLICY}" 
FEATURE_PREFIX = f"features_{P5_VER}_eegpsd_emgtd_mask_full_{MODE}_{QC_POLICY}"
SKIP_IF_EXISTS = True         # ✅ keep True; filenames are new so it won’t skip


SPLITS = ["train", "val", "test"]

# Folds to run:
FOLDS_TO_RUN: Optional[List[int]] = None  # e.g., [1,2,3] or None for ALL discovered folds

# If you re-ran Phase-5 (new exports), set this False (or delete old feature files)
SKIP_IF_EXISTS = True

# Signal constants
FS = 250.0
WELCH_NPERSEG = 256
EPS = 1e-8

# NumPy 2.x compatibility: trapz removed -> use trapezoid
try:
    _trapz = np.trapz
except AttributeError:
    _trapz = np.trapezoid


# STRICT mask-aware policy
MIN_VALID_FRAC    = 0.40        # per-channel validity fraction; below -> channel features zeroed
MIN_VALID_SAMPLES = 64          # PSD stability hard minimum

# Recommendation: compute features on de-normalized signals (using stats_fold.json)
USE_DENORM_FROM_STATS = True

EEG_BANDS: Dict[str, Tuple[float, float]] = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "mu":    (8.0, 13.0),
    "beta":  (13.0, 30.0),
}

# ---------------- OPTIONAL FILTERING ----------------
# IMPORTANT: For Phase-6 feature alignment, keep FILTER_MODE="all".
FILTER_MODE = "all"   # "all" | "tri_good" | "custom"
FILTER_KEPT_MODALITIES: Optional[int] = None   # set to 7 for tri-modal only (if key exists)
FILTER_QC_MIN: Optional[int] = None            # PASS=2 only -> 2; WARN+PASS -> 1
FILTER_QSCORE_MIN: Optional[float] = None      # e.g., 0.55


# =============================================================================
# Discovery + fold stats
# =============================================================================
def p55_discover_folds(prefix: str) -> List[int]:
    folds = []
    for p in DATASET_DIR.glob(f"{prefix}_fold*"):
        if not p.is_dir():
            continue
        try:
            folds.append(int(p.name.split("fold")[-1]))
        except ValueError:
            pass
    return sorted(set(folds))


def p55_load_fold_stats(export_prefix: str, fold_id: int) -> dict:
    """
    Loads per-fold mean/std from stats_fold.json.
    Returns dict with keys: "EEG", "EMG_env", "ET" (if present), each {mean,std}.
    """
    fold_dir = DATASET_DIR / f"{export_prefix}_fold{fold_id}"
    js_path = fold_dir / "stats_fold.json"
    if not js_path.exists():
        return {}
    js = json.loads(js_path.read_text())
    out = {}
    stats = js.get("stats", {})
    for k in ["EEG", "EMG_env", "ET"]:
        if k in stats and "mean" in stats[k] and "std" in stats[k]:
            out[k] = {
                "mean": np.asarray(stats[k]["mean"], dtype=np.float32),
                "std":  np.asarray(stats[k]["std"],  dtype=np.float32),
            }
    return out


def p55_denorm_to_nan(Xz: np.ndarray, M: Optional[np.ndarray], mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    """
    Xz: (T,C) z-scored (masked positions typically 0)
    M : (T,C) 0/1 mask
    Return denormed (T,C) with masked entries = NaN (so validity logic treats them invalid).
    """
    X = (Xz * std[None, :] + mean[None, :]).astype(np.float32, copy=False)
    if M is not None:
        X = X.copy()
        X[M <= 0.5] = np.nan
    return X


def p55_safe_log(x: np.ndarray, eps: float = EPS) -> np.ndarray:
    return np.log(np.maximum(x, eps))


def p55_all_zero_window(x: np.ndarray, tol: float = 1e-12) -> bool:
    if x.size == 0:
        return True
    with np.errstate(all="ignore"):
        return bool(np.nanmax(np.abs(x)) <= tol)


def p55_validity_check(m: np.ndarray, y: np.ndarray) -> Tuple[bool, float, int]:
    m = (np.asarray(m) > 0.5)
    y = np.asarray(y)
    valid = m & np.isfinite(y)
    cnt = int(valid.sum())
    frac = float(cnt) / float(max(1, y.shape[0]))
    ok = (cnt >= int(MIN_VALID_SAMPLES)) and (frac >= float(MIN_VALID_FRAC))
    return ok, frac, cnt


def p55_interp_fill_1d_strict(y: np.ndarray, m: np.ndarray) -> Tuple[np.ndarray, float]:
    y = np.asarray(y, np.float32)
    m = np.asarray(m, np.float32)

    ok, vfrac, _ = p55_validity_check(m, y)
    if not ok:
        return np.zeros_like(y, dtype=np.float32), vfrac

    m_bool = (m > 0.5) & np.isfinite(y)
    valid_idx = np.where(m_bool)[0]
    y_f = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)

    bad = ~m_bool
    if bad.any():
        xi = valid_idx.astype(np.float32)
        yi = y_f[valid_idx].astype(np.float32)
        x = np.arange(y.shape[0], dtype=np.float32)
        y_f[bad] = np.interp(x[bad], xi, yi).astype(np.float32)

    y_f = np.nan_to_num(y_f, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return y_f, vfrac


def p55_prepare_masked_window_strict(X: np.ndarray, M: Optional[np.ndarray]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    X = np.asarray(X, np.float32)
    T, C = X.shape
    if C == 0 or T == 0:
        return X, np.zeros((C,), dtype=np.float32), np.zeros((C,), dtype=np.float32)

    if M is None:
        M = np.ones_like(X, dtype=np.float32)
    else:
        M = np.asarray(M, np.float32)
        if M.shape != X.shape:
            raise RuntimeError(f"Mask shape mismatch: X{X.shape} vs M{M.shape}")

    X_out = np.zeros_like(X, dtype=np.float32)
    valid_chan = np.zeros((C,), dtype=np.float32)
    valid_frac = np.zeros((C,), dtype=np.float32)

    for c in range(C):
        yc, vfrac = p55_interp_fill_1d_strict(X[:, c], M[:, c])
        valid_frac[c] = float(vfrac)
        is_valid = (not p55_all_zero_window(yc)) and (vfrac >= MIN_VALID_FRAC)
        valid_chan[c] = 1.0 if is_valid else 0.0
        X_out[:, c] = yc

    return X_out, valid_chan, valid_frac


# =============================================================================
# EEG FEATURES
# =============================================================================
def p55_eeg_feats_one_window(x: np.ndarray, m: Optional[np.ndarray], fs: float) -> np.ndarray:
    x = np.asarray(x, np.float32)
    T, C = x.shape
    if T == 0 or C == 0:
        return np.zeros((0,), dtype=np.float32)
    if p55_all_zero_window(np.nan_to_num(x, nan=0.0)):
        return np.zeros((12 * C,), dtype=np.float32)

    x_f, valid_chan, _ = p55_prepare_masked_window_strict(x, m)

    f, Pxx = welch(
        x_f,
        fs=fs,
        axis=0,
        nperseg=min(WELCH_NPERSEG, T),
        noverlap=min(WELCH_NPERSEG, T) // 2,
        detrend="constant",
    )  # (F,C)

    m_1_30 = (f >= 1.0) & (f <= 30.0)
    if not np.any(m_1_30):
        return np.zeros((12 * C,), dtype=np.float32)

    f_1_30 = f[m_1_30]
    P_1_30 = Pxx[m_1_30]  # (F1,C)
    total_power = _trapz(P_1_30, x=f_1_30, axis=0) + EPS  # (C,)


    abs_band_feats, rel_band_feats = [], []
    for _, (lo, hi) in EEG_BANDS.items():
        mband = (f >= lo) & (f <= hi)
        if not np.any(mband):
            bp = np.zeros((C,), dtype=np.float32)
        else:
            bp = _trapz(Pxx[mband], x=f[mband], axis=0).astype(np.float32)

        abs_band_feats.append(p55_safe_log(bp))                 # log abs bandpower
        rel_band_feats.append((bp / total_power).astype(np.float32))

    abs_band = np.stack(abs_band_feats, axis=0)  # (4,C)
    rel_band = np.stack(rel_band_feats, axis=0)  # (4,C)

    # Spectral entropy over 1–30
    Psum = P_1_30.sum(axis=0, keepdims=True) + EPS
    Pnorm = P_1_30 / Psum
    spec_entropy = -(Pnorm * p55_safe_log(Pnorm)).sum(axis=0).astype(np.float32)  # (C,)
    spec_entropy_row = spec_entropy[None, :]

    # Hjorth on filled signal
    dx = np.diff(x_f, axis=0)
    ddx = np.diff(dx, axis=0)

    var_x = np.var(x_f, axis=0) + EPS
    var_dx = np.var(dx, axis=0) + EPS
    var_ddx = np.var(ddx, axis=0) + EPS

    activity = var_x
    mobility = np.sqrt(var_dx / var_x)
    mobility_dx = np.sqrt(var_ddx / var_dx)
    complexity = mobility_dx / (mobility + EPS)

    hj = np.stack([activity, mobility, complexity], axis=0).astype(np.float32)  # (3,C)
    hj = p55_safe_log(hj)

    all_feats = np.concatenate([abs_band, rel_band, spec_entropy_row, hj], axis=0)  # (12,C)
    all_feats *= valid_chan[None, :]  # hard-zero invalid channels
    return all_feats.reshape(-1).astype(np.float32)


# =============================================================================
# EMG FEATURES
# =============================================================================
def p55_zero_crossings(x: np.ndarray, thr: float = 0.0) -> int:
    if x.size < 2:
        return 0
    x1, x2 = x[:-1], x[1:]
    crosses = (x1 * x2) < 0
    if thr > 0.0:
        crosses = crosses & (np.abs(x2 - x1) >= thr)
    return int(crosses.sum())


def p55_ssc(x: np.ndarray, thr: float = 0.0) -> int:
    if x.size < 3:
        return 0
    x1, x2, x3 = x[:-2], x[1:-1], x[2:]
    cond = ((x2 - x1) * (x2 - x3)) > 0
    if thr > 0.0:
        cond = cond & ((np.abs(x2 - x1) >= thr) | (np.abs(x3 - x2) >= thr))
    return int(cond.sum())


def p55_emg_feats_one_window(x: np.ndarray, m: Optional[np.ndarray]) -> np.ndarray:
    x = np.asarray(x, np.float32)
    T, C = x.shape
    if T == 0 or C == 0:
        return np.zeros((0,), dtype=np.float32)
    if p55_all_zero_window(np.nan_to_num(x, nan=0.0)):
        return np.zeros((6 * C,), dtype=np.float32)

    x_f, valid_chan, _ = p55_prepare_masked_window_strict(x, m)

    feats = np.zeros((C, 6), dtype=np.float32)
    for c in range(C):
        if valid_chan[c] < 0.5:
            continue

        sig = x_f[:, c].astype(np.float32, copy=False)
        sig0 = sig - float(np.mean(sig))

        rms  = float(np.sqrt(np.mean(sig0 ** 2) + EPS))
        mav  = float(np.mean(np.abs(sig0)))
        wl   = float(np.sum(np.abs(np.diff(sig0))))
        zc   = float(p55_zero_crossings(sig0, thr=0.01))
        ssc  = float(p55_ssc(sig0, thr=0.01))
        varv = float(np.var(sig0))

        feats[c] = np.asarray([
            rms,
            mav,
            np.log(1.0 + max(wl, 0.0)),
            np.log(1.0 + max(zc, 0.0)),
            np.log(1.0 + max(ssc, 0.0)),
            np.log(1.0 + max(varv, 0.0)),
        ], dtype=np.float32)

    return feats.reshape(-1).astype(np.float32)


# =============================================================================
# Mask-fraction features
# =============================================================================
def p55_window_mask_fracs_from_keys(z, idx: int) -> Optional[np.ndarray]:
    keys = ("mask_frac_eeg", "mask_frac_emg", "mask_frac_et")
    if all(k in z.files for k in keys):
        return np.asarray([
            float(z["mask_frac_eeg"][idx]),
            float(z["mask_frac_emg"][idx]),
            float(z["mask_frac_et"][idx]),
        ], dtype=np.float32)
    return None


def p55_window_mask_fracs_from_masks(M_eeg, M_emg, M_et) -> np.ndarray:
    def _mean(M):
        if M is None:
            return 0.0
        M = np.asarray(M, np.float32)
        if M.size == 0:
            return 0.0
        return float(M.mean())
    return np.asarray([_mean(M_eeg), _mean(M_emg), _mean(M_et)], dtype=np.float32)


# =============================================================================
# Filtering policy
# =============================================================================
def p55_get_filter_settings() -> Tuple[Optional[int], Optional[int], Optional[float]]:
    mode = str(FILTER_MODE).lower().strip()
    if mode == "all":
        return None, None, None
    if mode == "tri_good":
        return 7, None, None
    if mode == "custom":
        return FILTER_KEPT_MODALITIES, FILTER_QC_MIN, FILTER_QSCORE_MIN
    print(f"[warn] Unknown FILTER_MODE='{FILTER_MODE}' -> defaulting to 'all'")
    return None, None, None


# =============================================================================
# Extraction
# =============================================================================
def _parse_shard_id_from_name(npz_path: Path) -> int:
    # expects names like train_shard_0001.npz / val_shard_0002.npz
    stem = npz_path.stem  # e.g., train_shard_0001
    parts = stem.split("_")
    if parts and parts[-1].isdigit():
        return int(parts[-1])
    # fallback: scan from end
    for p in reversed(parts):
        if p.isdigit():
            return int(p)
    return -1


def p55_extract_split_features(
    fold_id: int,
    split: str,
    export_prefix: str,
):
    fold_dir  = DATASET_DIR / f"{export_prefix}_fold{fold_id}"
    split_dir = fold_dir / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Split dir not found: {split_dir}")

    shard_paths = sorted(split_dir.glob("*_shard_*.npz"))
    if not shard_paths:
        raise FileNotFoundError(f"No shard npz files in {split_dir}")

    # ---- NEW: expected N from shards (alignment guard) ----
    expected_N = 0
    for sp0 in shard_paths:
        with np.load(sp0, allow_pickle=False) as z0:
            if "y_action" not in z0.files:
                raise RuntimeError(f"Shard missing y_action: {sp0}")
            expected_N += int(z0["y_action"].shape[0])

    # fold stats for de-normalization
    fold_stats = p55_load_fold_stats(export_prefix, fold_id) if USE_DENORM_FROM_STATS else {}
    eeg_stats = fold_stats.get("EEG", None)
    emg_stats = fold_stats.get("EMG_env", None)

    want_kept, want_qcmin, want_qmin = p55_get_filter_settings()

    eeg_list, emg_list, maskfeat_list = [], [], []
    meta_accum = defaultdict(list)

    # provenance (helps debugging and optional re-alignment)
    src_shard_id_parts = []
    src_row_in_shard_parts = []

    C_eeg_ref = None
    C_emg_ref = None

    print(f"[Fold {fold_id}][{split}] shards={len(shard_paths)} | prefix={export_prefix} | filter={FILTER_MODE} | denorm={USE_DENORM_FROM_STATS}")

    for sp in shard_paths:
        shard_id = _parse_shard_id_from_name(sp)

        # ---- CHANGED: allow_pickle=False (safer) ----
        with np.load(sp, allow_pickle=False) as z:
            if "X_EEG" not in z.files or "X_EMG" not in z.files:
                raise RuntimeError(f"Missing X_EEG/X_EMG in shard: {sp}")

            X_eeg = z["X_EEG"]
            X_emg = z["X_EMG"]

            M_eeg = z["M_EEG"] if "M_EEG" in z.files else None
            M_emg = z["M_EMG"] if "M_EMG" in z.files else None
            M_et  = z["M_ET"  ] if "M_ET"  in z.files else None

            N, T_eeg, C_eeg = X_eeg.shape
            _, T_emg, C_emg = X_emg.shape

            if C_eeg_ref is None: C_eeg_ref = C_eeg
            if C_emg_ref is None: C_emg_ref = C_emg
            if C_eeg != C_eeg_ref:
                raise RuntimeError(f"[Fold {fold_id}][{split}] EEG channels changed {C_eeg_ref}→{C_eeg} in {sp.name}")
            if C_emg != C_emg_ref:
                raise RuntimeError(f"[Fold {fold_id}][{split}] EMG channels changed {C_emg_ref}→{C_emg} in {sp.name}")

            # sanity for denorm
            if USE_DENORM_FROM_STATS:
                if eeg_stats is not None:
                    assert eeg_stats["mean"].shape[0] == C_eeg, f"EEG stats C mismatch: stats={eeg_stats['mean'].shape[0]} shard={C_eeg}"
                if emg_stats is not None:
                    assert emg_stats["mean"].shape[0] == C_emg, f"EMG stats C mismatch: stats={emg_stats['mean'].shape[0]} shard={C_emg}"

            # --------- keep mask (optional) ----------
            keep = np.ones((N,), dtype=bool)

            if want_kept is not None and "kept_modalities" in z.files:
                keep &= (np.asarray(z["kept_modalities"]) == int(want_kept))
            if want_qcmin is not None and "qc_flag" in z.files:
                keep &= (np.asarray(z["qc_flag"]) >= int(want_qcmin))
            if want_qmin is not None and "q_score" in z.files:
                keep &= (np.asarray(z["q_score"], np.float32) >= float(want_qmin))

            idxs = np.where(keep)[0]
            if idxs.size == 0:
                continue

            # provenance
            src_shard_id_parts.append(np.full((idxs.size,), int(shard_id), dtype=np.int32))
            src_row_in_shard_parts.append(idxs.astype(np.int32))

            # --------- carry meta ----------
            carry_keys = [
                "y_action", "y_task",
                "task_valid", "use_for_task",
                "subject_id", "task_code", "trial_id", "win_type", "win_len",
                "r_eeg", "r_emg", "r_et", "p_eeg", "p_emg", "p_et", "et_valid_frac",
                "mask_frac_eeg", "mask_frac_emg", "mask_frac_et",
                "kept_modalities", "qc_flag", "q_score", "sample_weight",
            ]
            for k in carry_keys:
                if k in z.files:
                    meta_accum[k].append(np.asarray(z[k])[idxs])

            # --------- featurize ----------
            for i in idxs:
                xi_eeg = np.asarray(X_eeg[i], np.float32)
                xi_emg = np.asarray(X_emg[i], np.float32)

                mi_eeg = None if M_eeg is None else np.asarray(M_eeg[i], np.float32)
                mi_emg = None if M_emg is None else np.asarray(M_emg[i], np.float32)
                mi_et  = None if M_et  is None else np.asarray(M_et[i],  np.float32)

                # De-normalize (recommended)
                if USE_DENORM_FROM_STATS:
                    if eeg_stats is not None:
                        xi_eeg = p55_denorm_to_nan(xi_eeg, mi_eeg, eeg_stats["mean"], eeg_stats["std"])
                    if emg_stats is not None:
                        xi_emg = p55_denorm_to_nan(xi_emg, mi_emg, emg_stats["mean"], emg_stats["std"])

                eeg_feat = p55_eeg_feats_one_window(xi_eeg, mi_eeg, fs=float(FS))
                emg_feat = p55_emg_feats_one_window(xi_emg, mi_emg)

                mf = p55_window_mask_fracs_from_keys(z, i)
                if mf is None:
                    mf = p55_window_mask_fracs_from_masks(mi_eeg, mi_emg, mi_et)

                eeg_list.append(eeg_feat)
                emg_list.append(emg_feat)
                maskfeat_list.append(mf)

    if len(eeg_list) == 0:
        raise FileNotFoundError(f"[Fold {fold_id}][{split}] After filtering, no windows left to featurize.")

    X_psd  = np.stack(eeg_list, axis=0).astype(np.float32)
    X_emgF = np.stack(emg_list, axis=0).astype(np.float32)
    X_mask = np.stack(maskfeat_list, axis=0).astype(np.float32)

    # ---- NEW: alignment guard when FILTER_MODE=all ----
    if str(FILTER_MODE).lower().strip() == "all":
        assert int(X_psd.shape[0]) == int(expected_N), \
            f"[align] expected N={expected_N} from shards but got N={X_psd.shape[0]} (did you filter?)"

    meta_out = {}
    for k, parts in meta_accum.items():
        meta_out[k] = np.concatenate(parts, axis=0)

    # add provenance
    meta_out["src_shard_id"] = np.concatenate(src_shard_id_parts, axis=0).astype(np.int32)
    meta_out["src_row_in_shard"] = np.concatenate(src_row_in_shard_parts, axis=0).astype(np.int32)

    # alignment sanity
    if "y_action" in meta_out:
        assert X_psd.shape[0] == meta_out["y_action"].shape[0], "Mismatch: features vs y_action length"

    info = {
        "C_eeg": int(C_eeg_ref or 0),
        "C_emg": int(C_emg_ref or 0),
        "D_eeg": int(X_psd.shape[1]),
        "D_emg": int(X_emgF.shape[1]),
        "D_mask": int(X_mask.shape[1]),
        "N": int(X_psd.shape[0]),
        "expected_N_from_shards": int(expected_N),
        "filter_mode": str(FILTER_MODE),
        "filter_kept_modalities": want_kept,
        "filter_qc_min": want_qcmin,
        "filter_q_score_min": want_qmin,
        "use_denorm_from_stats": bool(USE_DENORM_FROM_STATS),
        "has_provenance": True,
    }

    print(f"[Fold {fold_id}][{split}] X_psd={X_psd.shape} | X_emg={X_emgF.shape} | X_mask={X_mask.shape} | meta_keys={len(meta_out)}")
    return X_psd, X_emgF, X_mask, meta_out, info


def p55_run():
    print("========== Phase 5.5 (v6_3) — Feature Extraction ==========")
    print(f"Using EXPORT_PREFIX: {EXPORT_PREFIX}")
    folds = p55_discover_folds(EXPORT_PREFIX)
    if not folds:
        print(f"[error] No folds found for {EXPORT_PREFIX}_fold* under {DATASET_DIR}")
        return

    if FOLDS_TO_RUN is not None:
        want = set(int(x) for x in FOLDS_TO_RUN)
        folds = [f for f in folds if f in want]

    print(f"Folds to run: {folds}")

    for fid in folds:
        print(f"\n================== FOLD {fid} ==================")

        fold_meta = {
            "phase5_prefix": EXPORT_PREFIX,
            "feature_prefix": FEATURE_PREFIX,
            "fs": float(FS),
            "welch_nperseg": int(WELCH_NPERSEG),
            "min_valid_frac": float(MIN_VALID_FRAC),
            "min_valid_samples": int(MIN_VALID_SAMPLES),
            "use_denorm_from_stats": bool(USE_DENORM_FROM_STATS),
            "eeg_bands": EEG_BANDS,
            "eeg_feats_per_ch": 12,
            "emg_feats_per_ch": 6,
            "extra_feats": ["mask_frac_eeg", "mask_frac_emg", "mask_frac_et"],
            "strict_policy": "channels failing validity -> zero features",
            "filter_mode": str(FILTER_MODE),
            "provenance_fields": ["src_shard_id", "src_row_in_shard"],
        }

        for split in SPLITS:
            out_path = DATASET_DIR / f"{FEATURE_PREFIX}_fold{fid}_{split}.npz"
            if SKIP_IF_EXISTS and out_path.exists():
                print(f"[Fold {fid}][{split}] exists → skip ({out_path.name})")
                continue

            try:
                X_psd, X_emg, X_mask, meta_out, info = p55_extract_split_features(
                    fid, split, export_prefix=EXPORT_PREFIX
                )
            except FileNotFoundError as e:
                print(f"[Fold {fid}][{split}] skip: {e}")
                continue

            np.savez_compressed(
                out_path,
                X_psd=X_psd,
                X_emg=X_emg,
                X_mask=X_mask,
                meta_json=json.dumps({**fold_meta, **info}),
                **meta_out,
            )
            print(f"[Fold {fid}][{split}] saved → {out_path.name}")

    print("\n🎉 Phase 5.5 v6_3 complete!")


# Run
p55_run()


========== Phase 5.5 (v6_3) — Feature Extraction ==========
Using EXPORT_PREFIX: exports_v6_3c_commitmeta_balanced_core_ok
Folds to run: [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 30, 32, 33, 34, 35, 36]

================== FOLD 1 ==================
[Fold 1][train] shards=3 | prefix=exports_v6_3c_commitmeta_balanced_core_ok | filter=all | denorm=True
[Fold 1][train] X_psd=(12757, 96) | X_emg=(12757, 24) | X_mask=(12757, 3) | meta_keys=22
[Fold 1][train] saved → features_v6_3c_commitmeta_eegpsd_emgtd_mask_full_balanced_core_ok_fold1_train.npz
[Fold 1][val] shards=2 | prefix=exports_v6_3c_commitmeta_balanced_core_ok | filter=all | denorm=True
[Fold 1][val] X_psd=(8273, 96) | X_emg=(8273, 24) | X_mask=(8273, 3) | meta_keys=22
[Fold 1][val] saved → features_v6_3c_commitmeta_eegpsd_emgtd_mask_full_balanced_core_ok_fold1_val.npz
[Fold 1][test] shards=1 | prefix=exports_v6_3c_commitmeta_balanced_core_ok | filter=all | denorm=True
[Fold 1][test] X_

In [7]:
from pathlib import Path

DATASET_DIR = Path("/home/tsultan1/IROS/Data/_dataset_icml_v1")
fold = 35
for split in ["train","val","test"]:
    hits = sorted(DATASET_DIR.glob(f"features_v6_3c_commitmeta*fold{fold}_{split}.npz"))
    print(split, "hits:", len(hits))
    for h in hits[:5]:
        print(" ", h.name)


train hits: 1
  features_v6_3c_commitmeta_eegpsd_emgtd_mask_full_balanced_core_ok_fold35_train.npz
val hits: 1
  features_v6_3c_commitmeta_eegpsd_emgtd_mask_full_balanced_core_ok_fold35_val.npz
test hits: 1
  features_v6_3c_commitmeta_eegpsd_emgtd_mask_full_balanced_core_ok_fold35_test.npz
